# 

# Generalization Datasets

- Classificaiton
  - tox21
  - toxcast
  - muv
  - pcba
- Regression
  - hopv - homo, lumo
  - zinc15 - logp
  - freesolv - hydration free energy
- Rxn
  - open reaction database
- M2T
  - hanbum's dataset
- T2M
  - hanbum's dataset

# Check dataset availability

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm

def mol2graph(mol):
    """
    Converts SMILES string to graph Data object
    :input: SMILES string (str)
    :return: graph object
    """
    # atoms
    atom_features_list = []
    for atom in mol.GetAtoms():
        atom_features_list.append(atom_to_feature_vector(atom))
    x = np.array(atom_features_list, dtype = np.int64)

    # bonds
    num_bond_features = 3  # bond type, bond stereo, is_conjugated
    if len(mol.GetBonds()) > 0: # mol has bonds
        edges_list = []
        edge_features_list = []
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()

            edge_feature = bond_to_feature_vector(bond)

            # add edges in both directions
            edges_list.append((i, j))
            edge_features_list.append(edge_feature)
            edges_list.append((j, i))
            edge_features_list.append(edge_feature)

        # data.edge_index: Graph connectivity in COO format with shape [2, num_edges]
        edge_index = np.array(edges_list, dtype = np.int64).T

        # data.edge_attr: Edge feature matrix with shape [num_edges, num_edge_features]
        edge_attr = np.array(edge_features_list, dtype = np.int64)

    else:   # mol has no bonds
        edge_index = np.empty((2, 0), dtype = np.int64)
        edge_attr = np.empty((0, num_bond_features), dtype = np.int64)

    graph = dict()
    graph['edge_index'] = edge_index
    graph['edge_feat'] = edge_attr
    graph['node_feat'] = x
    graph['num_nodes'] = len(x)

    return graph 

from rdkit import Chem
import selfies as sf
from download_dataset import wrap_label

system_prompt = "You are a helpful assistant for molecular chemistry, to address tasks including molecular property classification, molecular property regression, chemical reaction prediction, molecule captioning, molecule generation."

def prepare_data_instance(
        mol,
        label,
        task,
        instruction_templates,
        system_prompt,
        mol_token="<mol>",
        num_query_tokens=32,
):
    smiles = Chem.MolToSmiles(mol)
    selfies = sf.encoder(smiles)
    input_mol_string = "<SELFIES> " + selfies + " </SELFIES>"

    label = wrap_label(label, task=task)
    graph = mol2graph(mol)
    
    input_prompt = np.random.choice(instruction_templates).item()


    graph_sequence = "<GRAPH>" + mol_token * num_query_tokens + "</GRAPH>"
    input_mol_string_graph = input_mol_string + graph_sequence
    assert "<INPUT>" in input_prompt, f"llm_prompt should contain <INPUT>"

    input_prompt = input_prompt.replace("<INPUT>", input_mol_string_graph)

    formatted_prompt_text = "<s>[INST] " + system_prompt + " \n\n" + input_prompt + " [INST]"
    formatted_target_text = label + " </s>"

    data = {
        "task": task,
        "x": graph['node_feat'],
        "edge_index": graph['edge_index'],
        "edge_attr": graph['edge_feat'],
        "additional_x": graph['node_feat'],
        "additional_edge_index": graph['edge_index'],
        "additional_edge_attr": graph['edge_feat'],
        "input_mol_string": input_mol_string,
        "prompt_text": formatted_prompt_text,
        "target_text": formatted_target_text,
    }
    return data


def get_data_list(
        list_mol, list_label, task, instruction_templates, system_prompt
):
    list_data = []
    iter_bar = tqdm(range(len(list_mol)))

    for i in iter_bar:
        data = prepare_data_instance(
        mol=list_mol[i],
        label=list_label[i], 
        task=task,
        instruction_templates=instruction_templates,
        system_prompt=system_prompt
    )  
        list_data.append(data)
    return list_data


No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


In [2]:
import deepchem as dc
task_name = "hopv"
base_path = f"/text-mol/dataset/{task_name}"

tasks, hopv_dataset, transformers = dc.molnet.load_hopv(
                featurizer="Raw",
                splitter="scaffold",
                save_dir=base_path,
                data_dir=base_path,
                reload=True,
            )

In [3]:
import instructions_smol
import model.added_tokens as added_tokens
import numpy as np

# class arugments
list_mol = [hopv_dataset[i].X for i in range(3)]
list_mol = np.concatenate(list_mol)
list_y = [hopv_dataset[i].y for i in range(3)]
list_y = np.concatenate(list_y)

In [4]:
import datasets

subtask_id = 0
task_name = "hopv_homo"
list_label = list_y[:, subtask_id]
instruction_templates = instructions_smol.qm9_homo

list_homo = get_data_list(
    list_mol=list_mol,
    list_label=list_label, 
    task=task_name, 
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
hopv_homo_dataset = datasets.Dataset.from_list(list_homo)

100%|██████████| 350/350 [00:00<00:00, 460.27it/s]


In [5]:
subtask_id = 1
task_name = "hopv_lumo"
list_label = list_y[:, subtask_id]
instruction_templates = instructions_smol.qm9_lumo

list_lumo = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
hopv_lumo_dataset = datasets.Dataset.from_list(list_lumo)

100%|██████████| 350/350 [00:00<00:00, 509.68it/s]


In [6]:
llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

In [7]:
hopv_lumo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_hopv_lumo")
hopv_lumo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_hopv_lumo")
hopv_lumo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_hopv_lumo")

hopv_homo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_hopv_homo")
hopv_homo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_hopv_homo")
hopv_homo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_hopv_homo")

Saving the dataset (1/1 shards): 100%|██████████| 350/350 [00:00<00:00, 32836.16 examples/s]


In [10]:
hopv_homo_dataset[0]

{'task': 'hopv_homo',
 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [15, 0, 2, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [15, 0, 2, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [15, 0, 2, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [15, 0, 2, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [7, 0, 1, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 3, 5, 1, 0, 1, 1, 1],
  [5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1]],
 'edge_index': [[0,
   23,
   23,
   16,
   23,
   2,
   2,
   19,
   19,
   7,
   7,
   4,
   4,
   18,
   18,
   11,
   11,
   8,
   8,
   24,
   24,
   20,
   20,
   14,
   14,
   1,
   1,
 